# Занятие 1. Python как лабораторный инструмент

**Вычислительная физика с Python** · мини-курс для 10–11 классов

Сегодня:
1. Jupyter Lab и основы Python
2. Массивы `numpy`: считаем сразу тысячи значений
3. Графики `matplotlib`
4. Обработка эксперимента: метод наименьших квадратов, постоянная Планка из опыта Милликена
5. Интерполяция: что происходит между точками
6. Мини-эксперимент: измеряем $g$ маятником

## 0. Как работать в Jupyter Lab

* Ноутбук состоит из **ячеек**: в текстовых — пояснения и формулы, в ячейках кода — программы на Python.
* `Shift + Enter` — выполнить ячейку и перейти к следующей.
* `A` / `B` (вне режима редактирования) — вставить ячейку выше / ниже, `M` — сделать ячейку текстовой, `Y` — ячейкой кода.
* Важен **порядок выполнения**, а не порядок на экране: число в `[ ]` слева показывает, какой по счёту выполнилась ячейка. Если запутались — меню *Kernel → Restart Kernel and Run All Cells*.
* Справка по функции: `help(имя)` или `имя?`.

## 1. Python как калькулятор физика

Переменные, арифметика, степень `**`, научная запись чисел `6.63e-34`.

In [ ]:
g = 9.81          # ускорение свободного падения, м/с^2
h = 20.0          # высота, м

t = (2 * h / g) ** 0.5    # время падения
v = g * t                 # скорость у земли

print('Время падения:', t, 'с')
print(f'Скорость у земли: {v:.1f} м/с = {v * 3.6:.0f} км/ч')

Конструкция `f'...{v:.1f}...'` (f-строка) подставляет значение переменной в текст; `:.1f` — один знак после запятой, `:.3e` — научная запись.

Для математических функций нужна библиотека. Мы сразу будем пользоваться `numpy` — в ней есть и функции, и массивы.

In [ ]:
import numpy as np

print(np.pi, np.sqrt(2), np.sin(np.pi / 6), np.exp(1))

### Функции

Повторяющийся расчёт оформляем в функцию. Период малых колебаний математического маятника:
$$T = 2\pi\sqrt{\frac{L}{g}}.$$

In [ ]:
def pendulum_period(L, g=9.81):
    '''Период малых колебаний маятника длиной L (м).'''
    return 2 * np.pi * np.sqrt(L / g)

print(pendulum_period(1.0))
print(pendulum_period(1.0, g=1.62))   # на Луне

### Циклы и условия

In [ ]:
for L in [0.25, 0.5, 1.0, 2.0, 4.0]:
    T = pendulum_period(L)
    if T > 2:
        note = '(дольше 2 с)'
    else:
        note = ''
    print(f'L = {L:4} м   T = {T:.2f} с {note}')

Заметьте: когда длина растёт в 4 раза, период растёт в 2 раза. Это и есть $T \propto \sqrt{L}$.

### Задание 1 — решение

In [ ]:
def fall(h, g=9.81):
    t = np.sqrt(2 * h / g)
    return t, g * t

t, v = fall(300)
print(f't = {t:.1f} с, v = {v:.0f} м/с = {v * 3.6:.0f} км/ч')

## 2. Массивы numpy: считаем всё сразу

Массив `numpy` — это набор чисел, с которым можно обращаться как с одним числом: арифметика применяется **к каждому элементу**.

In [ ]:
L = np.array([0.25, 0.5, 1.0, 2.0, 4.0])
T = pendulum_period(L)     # та же функция, но сразу для всего массива
print(T)
print(T**2 / L)            # должно быть 4*pi^2/g для всех элементов
print(4 * np.pi**2 / 9.81)

Полезные способы создать массив:
* `np.linspace(a, b, n)` — `n` равноотстоящих точек от `a` до `b` включительно;
* `np.arange(a, b, step)` — от `a` до `b` (не включая) с шагом `step`;
* `np.zeros(n)`, `np.ones(n)`.

Индексы начинаются с нуля, `a[-1]` — последний элемент, `a[2:5]` — срез.

In [ ]:
t = np.linspace(0, 2, 5)
print(t)
print(t[0], t[-1], t[1:3])
print(len(t), t.sum(), t.mean(), t.max())

### Почему это быстро

Сравним цикл Python и операцию над массивом для миллиона точек.

In [ ]:
import time

N = 1_000_000
x = np.linspace(0, 10, N)

start = time.perf_counter()
y_loop = [np.sin(xi) ** 2 for xi in x]
t_loop = time.perf_counter() - start

start = time.perf_counter()
y_np = np.sin(x) ** 2
t_np = time.perf_counter() - start

print(f'цикл:  {t_loop:.3f} с')
print(f'numpy: {t_np:.4f} с   — быстрее в {t_loop / t_np:.0f} раз')

### Задание 2 — решение

In [ ]:
m = 1500
v_kmh = np.arange(0, 121, 10)
E = m * (v_kmh / 3.6) ** 2 / 2 / 1000     # кДж
for vi, Ei in zip(v_kmh, E):
    print(f'{vi:4d} км/ч  {Ei:7.1f} кДж')
print('E(120) / E(60) =', E[-1] / E[6])

## 3. Графики matplotlib

Хороший физический график: подписи осей **с единицами**, легенда, если кривых несколько, сетка по желанию.

In [ ]:
import matplotlib.pyplot as plt

g = 9.81
v0 = 15.0                        # начальная скорость вверх, м/с
t = np.linspace(0, 2 * v0 / g, 100)
y = v0 * t - g * t**2 / 2
v = v0 - g * t

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(t, y)
ax1.set_xlabel('$t$, с')
ax1.set_ylabel('$y$, м')
ax1.set_title('Координата')
ax1.grid(alpha=0.3)

ax2.plot(t, v, color='C1')
ax2.axhline(0, color='k', lw=0.8)
ax2.set_xlabel('$t$, с')
ax2.set_ylabel('$v_y$, м/с')
ax2.set_title('Скорость')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Подписи в `$...$` пишутся в формате LaTeX, как формулы в учебнике.

### Простое правило — сложное поведение

Логистическое отображение $x_{n+1} = r\,x_n(1-x_n)$ — простейшая модель численности популяции. При $r < 3$ численность устанавливается, затем начинает колебаться между 2, 4, 8… значениями, а при $r \approx 3{,}57$ наступает **хаос**. Построим, какие значения $x$ получаются после долгого счёта при каждом $r$ — сразу для 2000 значений $r$ с помощью массивов.

In [ ]:
r = np.linspace(2.5, 4.0, 2000)
x = 0.5 * np.ones_like(r)

for n in range(500):          # ждём, пока установится режим
    x = r * x * (1 - x)

plt.figure(figsize=(10, 5))
for n in range(200):          # рисуем следующие 200 значений
    x = r * x * (1 - x)
    plt.plot(r, x, ',k', alpha=0.3)
plt.xlabel('$r$')
plt.ylabel('$x$')
plt.title('Бифуркационная диаграмма логистического отображения')
plt.show()

### Задание 3 — решение

In [ ]:
v0, g = 20.0, 9.81
for deg in [30, 45, 60]:
    a = np.radians(deg)
    t = np.linspace(0, 2 * v0 * np.sin(a) / g, 100)
    plt.plot(v0 * np.cos(a) * t, v0 * np.sin(a) * t - g * t**2 / 2, label=f'{deg}°')
plt.xlabel('$x$, м')
plt.ylabel('$y$, м')
plt.axis('equal')
plt.grid(alpha=0.3)
plt.legend()
plt.show()
# Наибольшая дальность при 45°, у 30° и 60° дальность одинакова: sin(2a) совпадает.

## 4. Обработка эксперимента: постоянная Планка из опыта Милликена

Уравнение Эйнштейна для фотоэффекта: $h\nu = A + \dfrac{m v^2}{2}$. Кинетическую энергию самых быстрых электронов измеряют по **запирающему напряжению** $U_з$: $\;eU_з = \dfrac{mv^2}{2}$. Отсюда

$$U_з = \frac{h}{e}\,\nu - \frac{A}{e}.$$

Зависимость $U_з(\nu)$ — прямая, а её наклон $k = h/e$. Роберт Милликен поставил этот опыт (опубликован в 1916 г.), сомневаясь в гипотезе фотонов, — и подтвердил формулу Эйнштейна.

Файл `data/millikan.txt`: в первом столбце частота света $\nu$ (Гц), во втором — запирающее напряжение $U_з$ (В).

In [ ]:
nu, U = np.loadtxt('../../data/millikan.txt', unpack=True)
print(nu)
print(U)

plt.plot(nu, U, 'o')
plt.xlabel(r'$\nu$, Гц')
plt.ylabel('$U_з$, В')
plt.grid(alpha=0.3)
plt.show()

### Метод наименьших квадратов (МНК)

Точки почти на прямой, но не точно. Какую прямую $U = k\nu + b$ провести «лучше всего»? МНК выбирает $k$ и $b$ так, чтобы **сумма квадратов отклонений** точек от прямой была минимальной:
$$S(k, b) = \sum_{i=1}^{N} \left(U_i - k\nu_i - b\right)^2 \to \min.$$
Приравнивая к нулю производные $S$ по $k$ и по $b$, получаем:
$$k = \frac{\langle \nu U\rangle - \langle \nu\rangle\langle U\rangle}{\langle \nu^2\rangle - \langle \nu\rangle^2}, \qquad b = \langle U\rangle - k\langle \nu\rangle,$$
где $\langle\ldots\rangle$ — среднее по всем точкам.

In [ ]:
k = (np.mean(nu * U) - np.mean(nu) * np.mean(U)) / (np.mean(nu**2) - np.mean(nu)**2)
b = np.mean(U) - k * np.mean(nu)

e = 1.602176634e-19          # заряд электрона, Кл
h = e * k
print(f'k = {k:.4e} В/Гц,  b = {b:.3f} В')
print(f'h = {h:.3e} Дж·с   (табличное значение 6.626e-34)')
print(f'отклонение: {abs(h - 6.62607015e-34) / 6.62607015e-34 * 100:.1f} %')

То же самое одной строкой: `np.polyfit(x, y, 1)` подбирает полином 1-й степени (прямую) методом наименьших квадратов.

In [ ]:
k2, b2 = np.polyfit(nu, U, 1)
print(k2, b2)

nn = np.linspace(4e14, 12.5e14, 2)
plt.plot(nu, U, 'o', label='данные Милликена')
plt.plot(nn, k2 * nn + b2, label='прямая МНК')
plt.xlabel(r'$\nu$, Гц')
plt.ylabel('$U_з$, В')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### С какой точностью мы знаем $h$?

`scipy.optimize.curve_fit` подбирает параметры **любой** функции и заодно оценивает их погрешности: корни из диагонали ковариационной матрицы `pcov`.

In [ ]:
from scipy.optimize import curve_fit

def line(x, k, b):
    return k * x + b

popt, pcov = curve_fit(line, nu, U)
k_fit, b_fit = popt
dk, db = np.sqrt(np.diag(pcov))

print(f'h = ({e * k_fit * 1e34:.2f} ± {e * dk * 1e34:.2f})·10^-34 Дж·с')

### Задание 4 — решение

In [ ]:
A_eV = -b
nu0 = -b / k
lam0 = 3e8 / nu0
print(f'A = {A_eV:.2f} эВ, nu0 = {nu0:.3e} Гц, lambda0 = {lam0 * 1e9:.0f} нм')
# ~710 нм — граница видимого диапазона (красный свет)

plt.plot(nu, U - (k * nu + b), 'o')
plt.axhline(0, color='k', lw=0.8)
plt.xlabel(r'$\nu$, Гц')
plt.ylabel('отклонение, В')
plt.grid(alpha=0.3)
plt.show()

### Линеаризация

МНК для прямой — самый простой и надёжный. Поэтому нелинейную зависимость часто **превращают в прямую** подходящей заменой переменных. Для маятника
$$T = 2\pi\sqrt{\frac{L}{g}} \quad\Longrightarrow\quad T^2 = \frac{4\pi^2}{g}\,L,$$
то есть $T^2$ линейно зависит от $L$, а по наклону находим $g$.

Ниже — пример измерений (числа придуманы для примера, со случайной ошибкой порядка 0,01 с). На мини-эксперименте вы подставите свои.

In [ ]:
L_data = np.array([0.20, 0.35, 0.50, 0.65, 0.80, 1.00])    # м
T_data = np.array([0.91, 1.18, 1.43, 1.61, 1.80, 2.00])    # с

### Задание 5 — решение

In [ ]:
kT, bT = np.polyfit(L_data, T_data**2, 1)
g_exp = 4 * np.pi**2 / kT
print(f'g = {g_exp:.2f} м/с^2, свободный член b = {bT:.3f} с^2 (должен быть близок к 0)')

LL = np.linspace(0, 1.1, 2)
plt.plot(L_data, T_data**2, 'o', label='измерения')
plt.plot(LL, kT * LL + bT, label=f'МНК: g = {g_exp:.2f} м/с$^2$')
plt.xlabel('$L$, м')
plt.ylabel('$T^2$, с$^2$')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 5. Интерполяция: что между точками?

Часто физическая величина известна только **таблицей**. Давление насыщенного водяного пара (справочные значения):

In [ ]:
T_tab = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])                         # °C
p_tab = np.array([0.611, 1.228, 2.339, 4.246, 7.384, 12.35, 19.95, 31.18, 47.41, 70.18, 101.3])  # кПа

plt.plot(T_tab, p_tab, 'o')
plt.xlabel('$t$, °C')
plt.ylabel('$p_{нас}$, кПа')
plt.grid(alpha=0.3)
plt.show()

Каково давление при температуре тела 37 °C? Проще всего соединить соседние точки отрезками — **линейная интерполяция** (`np.interp`). Гладкая кривая через все точки, составленная из кубических кусочков, — **кубический сплайн** (`scipy.interpolate.CubicSpline`).

Для проверки сравним с эмпирической формулой Магнуса $p \approx 0{,}6109\,\exp\!\left(\dfrac{17{,}625\,t}{t + 243{,}04}\right)$ кПа.

In [ ]:
from scipy.interpolate import CubicSpline

spline = CubicSpline(T_tab, p_tab)
magnus = lambda t: 0.61094 * np.exp(17.625 * t / (t + 243.04))

t0 = 37
print(f'линейная:  {np.interp(t0, T_tab, p_tab):.3f} кПа')
print(f'сплайн:    {spline(t0):.3f} кПа')
print(f'Магнус:    {magnus(t0):.3f} кПа')

tt = np.linspace(0, 100, 300)
plt.plot(T_tab, p_tab, 'o', label='таблица')
plt.plot(tt, np.interp(tt, T_tab, p_tab), label='линейная')
plt.plot(tt, spline(tt), '--', label='сплайн')
plt.xlabel('$t$, °C')
plt.ylabel('$p_{нас}$, кПа')
plt.xlim(20, 60)
plt.ylim(0, 25)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### Осторожно: полином через все точки

Кажется естественным провести через $N$ точек полином степени $N-1$ — он пройдёт через все точки точно. Но между точками он может сильно «раскачиваться» (эффект Рунге, 1901 г.):

In [ ]:
f = lambda x: 1 / (1 + 25 * x**2)
xn = np.linspace(-1, 1, 11)
xx = np.linspace(-1, 1, 500)

p = np.polyfit(xn, f(xn), len(xn) - 1)

plt.plot(xx, f(xx), 'k', lw=2, label='функция')
plt.plot(xx, np.polyval(p, xx), label='полином 10-й степени')
plt.plot(xx, CubicSpline(xn, f(xn))(xx), '--', label='сплайн')
plt.plot(xn, f(xn), 'ok')
plt.ylim(-0.5, 2)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

**Вывод:** для интерполяции таблиц используют кусочные функции (сплайны), а для экспериментальных данных с ошибками — не интерполяцию, а **подгонку** простой физически обоснованной зависимости (МНК).

### Задание 6 — решение

In [ ]:
p0 = 50.0
print(f'линейная: {np.interp(p0, p_tab, T_tab):.1f} °C')
print(f'сплайн:   {CubicSpline(p_tab, T_tab)(p0):.1f} °C')
# около 81 °C

## 6. Мини-эксперимент: ускорение свободного падения

**Оборудование:** нитка, гайка или другой небольшой груз, линейка или рулетка, секундомер (телефон).

1. Измерьте время $t_{10}$ десяти полных колебаний для 5–6 разных длин нити от 20 см до 1 м. Амплитуда — небольшая (отклонение на 5–10 см).
2. Длину измеряйте от точки подвеса до **центра** груза.
3. Внесите данные в ячейку ниже, постройте $T^2(L)$ и найдите $g$ по наклону.
4. Обсудите: почему измеряют 10 колебаний, а не одно? Почему прямая может не проходить через ноль?

*Для желающих:* приложение **phyphox** (бесплатное, iOS и Android) измеряет период маятника по гироскопу телефона и экспортирует данные в CSV.

In [ ]:
L_my = np.array([])      # длины, м
t10_my = np.array([])    # время 10 колебаний, с

T_my = t10_my / 10
# дальше — как в задании 5

## Домашнее задание: солнечный цикл

Файл `data/sunspots.txt` — среднемесячное число солнечных пятен (числа Вольфа) начиная с января 1749 г. Первый столбец — номер месяца, второй — число пятен.

1. Загрузите данные и постройте график (по оси $x$ — годы: `1749 + месяц/12`).
2. Сгладьте данные скользящим средним за 12 месяцев: `np.convolve(s, np.ones(12)/12, mode='same')`.
3. Оцените средний период солнечной активности. *Подсказка:* найдите максимумы с помощью `scipy.signal.find_peaks`. Параметр `distance` задаёт минимальное расстояние между пиками (в отсчётах), а `prominence` отбрасывает мелкие «зубцы» на склонах. Проверьте на графике, что найдены именно максимумы циклов.

На последних занятиях мы научимся находить период с помощью преобразования Фурье.

### Почитать
* Дж. Тейлор. *Введение в теорию ошибок.* — как правильно оценивать погрешности.
* Дж. Элленберг. *Как не ошибаться. Сила математического мышления.*

### Домашнее задание — решение

In [ ]:
from scipy.signal import find_peaks

month, s = np.loadtxt('../../data/sunspots.txt', unpack=True)
year = 1749 + month / 12
s_smooth = np.convolve(s, np.ones(12) / 12, mode='same')

peaks, _ = find_peaks(s_smooth, distance=7 * 12, prominence=20)
period = np.diff(year[peaks])
print(f'найдено максимумов: {len(peaks)}')
print(f'средний период: {period.mean():.1f} ± {period.std():.1f} лет')

# около 11 лет, но отдельные циклы заметно различаются по длине

plt.figure(figsize=(12, 4))
plt.plot(year, s, lw=0.5, alpha=0.5, label='по месяцам')
plt.plot(year, s_smooth, label='скользящее среднее, 12 мес.')
plt.plot(year[peaks], s_smooth[peaks], 'v', color='C3', label='максимумы')
plt.xlabel('год')
plt.ylabel('число Вольфа')
plt.legend()
plt.show()